In [2]:
import torch
import torch.nn as nn
from torch.nn import functional as F
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)
block_size=8
batch_size=4

cpu


In [3]:
with open('wizard_of_oz.txt', 'r', encoding='utf-8') as f:
    text = f.read()
chars = sorted(set(text))
print(chars)
vocab_size = len(chars)

['\n', ' ', '!', '"', '&', "'", '(', ')', '*', ',', '-', '.', '0', '1', '2', '3', '4', '5', '6', '7', '8', '9', ':', ';', '?', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', '[', ']', '_', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z', '\ufeff']


In [4]:
string_to_int = { ch:i for i,ch in enumerate(chars) }
int_to_string = { i:ch for i,ch in enumerate(chars) }
encode = lambda s: [string_to_int[c] for c in s]
decode = lambda l: ''.join([int_to_string[i] for i in l])

data = torch.tensor(encode(text), dtype=torch.long)
print(data[:100])


tensor([80,  1,  1, 28, 39, 42, 39, 44, 32, 49,  1, 25, 38, 28,  1, 44, 32, 29,
         1, 47, 33, 50, 25, 42, 28,  1, 33, 38,  1, 39, 50,  0,  0,  1,  1, 26,
        49,  0,  0,  1,  1, 36, 11,  1, 30, 42, 25, 38, 35,  1, 26, 25, 45, 37,
         0,  0,  1,  1, 25, 45, 44, 32, 39, 42,  1, 39, 30,  1, 44, 32, 29,  1,
        47, 33, 50, 25, 42, 28,  1, 39, 30,  1, 39, 50,  9,  1, 44, 32, 29,  1,
        36, 25, 38, 28,  1, 39, 30,  1, 39, 50])


In [5]:
n = int(0.8*len(data))
train_data = data[:n]
val_data = data[n:]

def get_batch(split):
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    x, y = x.to(device), y.to(device)
    return x, y

x, y = get_batch('train')
print('inputs:')
# print(x.shape)
print(x)
print('targets:')
print(y)


inputs:
tensor([[ 1, 54, 67, 57,  1, 54, 65, 65],
        [72, 68, 71, 73, 11,  1, 44, 61],
        [65, 68, 68, 64, 58, 57,  1, 65],
        [61, 58,  1, 60, 65, 54, 72, 72]])
targets:
tensor([[54, 67, 57,  1, 54, 65, 65, 68],
        [68, 71, 73, 11,  1, 44, 61, 58],
        [68, 68, 64, 58, 57,  1, 65, 62],
        [58,  1, 60, 65, 54, 72, 72,  1]])


In [6]:
x = train_data[:block_size]
y = train_data[:block_size+1]

for t in range(block_size):
    context = x[:t+1]
    target = y[t]
    print("when input is", context, "target is", target)

when input is tensor([80]) target is tensor(80)
when input is tensor([80,  1]) target is tensor(1)
when input is tensor([80,  1,  1]) target is tensor(1)
when input is tensor([80,  1,  1, 28]) target is tensor(28)
when input is tensor([80,  1,  1, 28, 39]) target is tensor(39)
when input is tensor([80,  1,  1, 28, 39, 42]) target is tensor(42)
when input is tensor([80,  1,  1, 28, 39, 42, 39]) target is tensor(39)
when input is tensor([80,  1,  1, 28, 39, 42, 39, 44]) target is tensor(44)


In [7]:
class BigramLanguageModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)
        
    def forward(self, index, targets=None):
        logits = self.token_embedding_table(index)
        
        
        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)
        
        return logits, loss
    
    def generate(self, index, max_new_tokens):
        for _ in range(max_new_tokens):
            logits, loss = self.forward(index)
            logits = logits[:, -1, :] 
            probs = F.softmax(logits, dim=-1) 
            index_next = torch.multinomial(probs, num_samples=1)
            index = torch.cat((index, index_next), dim=1) 
        return index

model = BigramLanguageModel(vocab_size)
m = model.to(device)

context = torch.zeros((1,1), dtype=torch.long, device=device)
generated_chars = decode(m.generate(context, max_new_tokens=500)[0].tolist())
print(generated_chars)



c'RAGLrALe'RpKs﻿S_&NXc_Us,z::&,43CQL]ffWXuy)kL
[&QL5u0gomKEeaN2uJ
(bU4z.rgiIy
!zI"S6
*5wmPg0xn,(m2F!rP222s('dav;P_EwGcyi[p&L;0RgdaOL﻿4qSBQnJ'fd;b"wCr!xDdMVXMznbYn]j'R?9uUA119]BHN)u19;f?mK"7﻿RCAHu2.tk.Ba4exb﻿eH!YJ4jyTMFGLr?wl﻿gF
6?RrY
LGPL5F!C CY BvXcz5]-v[g [xb﻿PYSv  vWvPZqCh9AL5ZZ"BmGln3cuRxr)zI_5KhWlS64Fon'Cw5I;DSBxhjoHmJ40:OPsqO?i
&I!H00WEob"YMzI8w!,]:"S'C*Ju!yGLIsdlYn Cn znVEQ;fA]fAGL5P2rcxN.&]x)*-o,5:-W
DTZ[plJ
)]c8PC32FOzn)yMO*h_lS
qgG"Eo!N68D)0:p5::GtfTOxu5FWrxDz_jzsgR6ASc0:vN 1-DWwftCr﻿6
